In [1]:
!ls -lh

total 26M
-rw-r--r-- 1 root root 3.0M Jun 10 17:34 GCF_006345805.1_ASM634580v1_feature_table.txt
drwxr-xr-x 1 root root 4.0K Jun  4 13:32 sample_data
-rw-r--r-- 1 root root  23M Jun 10 17:34 zhunt.zip


In [6]:
!unzip -q zhunt.zip -d zhunt
!find zhunt -type f | head

replace zhunt/LG23_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG24_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG25_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG26_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG27_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG28_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG29_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG30_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG1_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG2_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG3_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG4_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG6_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG7_zhunt.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace zhunt/LG8_zhunt.cs

Результаты предсказания Z-ДНК были получены отдельно для каждой linkage group генома. На данном этапе производится чтение и объединение всех файлов в единую таблицу

In [8]:
import pandas as pd
from pathlib import Path

csv_files = sorted(Path("zhunt").rglob("*.csv"))

print("CSV files:", len(csv_files))
print(csv_files[:5])

test = pd.read_csv(csv_files[0], sep="\t")

print(test.shape)
test.head()

CSV files: 30
[PosixPath('zhunt/LG10_zhunt.csv'), PosixPath('zhunt/LG11_zhunt.csv'), PosixPath('zhunt/LG12_zhunt.csv'), PosixPath('zhunt/LG13_zhunt.csv'), PosixPath('zhunt/LG14_zhunt.csv')]
(72918, 8)


,ID,POSITION,LENGTH,SEQUENCE,ZDNAGCRICHNESS,ZDNAGTRICHNESS,SCORE,SCOREPERC
0,1,9629,51,GCACACACACACACACACACACACACACACACACACACACACACAC...,2.0,0.0,86.0,13.76
1,2,12275,23,ACACACACACACACACACACACA,0.0,0.0,33.0,12.00
2,3,12544,40,CACACACACACACACACACACACACACACACACACACACA,0.0,0.0,58.5,12.00
3,4,12617,17,ACACACACACACGCACA,12.5,0.0,46.0,23.00
4,5,12654,13,ACACACACACACA,0.0,0.0,18.0,12.00


Для каждой linkage group добавляются координаты хромосом RefSeq и рассчитываются координаты участков в форматах 1-based и 0-based

In [9]:
import pandas as pd
from pathlib import Path

lg_to_accession = {
    "LG1": "NC_042997.1",
    "LG2": "NC_042998.1",
    "LG3": "NC_042999.1",
    "LG4": "NC_043000.1",
    "LG5": "NC_043001.1",
    "LG6": "NC_043002.1",
    "LG7": "NC_043003.1",
    "LG8": "NC_043004.1",
    "LG9": "NC_043005.1",
    "LG10": "NC_043006.1",
    "LG11": "NC_043007.1",
    "LG12": "NC_043008.1",
    "LG13": "NC_043009.1",
    "LG14": "NC_043010.1",
    "LG15": "NC_043011.1",
    "LG16": "NC_043012.1",
    "LG17": "NC_043013.1",
    "LG18": "NC_043014.1",
    "LG19": "NC_043015.1",
    "LG20": "NC_043016.1",
    "LG21": "NC_043017.1",
    "LG22": "NC_043018.1",
    "LG23": "NC_043019.1",
    "LG24": "NC_043020.1",
    "LG25": "NC_043021.1",
    "LG26": "NC_043022.1",
    "LG27": "NC_043023.1",
    "LG28": "NC_043024.1",
    "LG29": "NC_043025.1",
    "LG30": "NC_043026.1",
}

dfs = []

for file in sorted(Path("zhunt").glob("*.csv")):
    df = pd.read_csv(file, sep="\t")
    df.columns = df.columns.str.replace('"', '', regex=False).str.strip()

    lg = file.stem.replace("_zhunt", "")
    df["LG"] = lg
    df["chrom"] = lg_to_accession[lg]

    df["POSITION"] = pd.to_numeric(df["POSITION"])
    df["LENGTH"] = pd.to_numeric(df["LENGTH"])

    df["start_1based"] = df["POSITION"]
    df["end_1based"] = df["POSITION"] + df["LENGTH"] - 1

    df["start_0based"] = df["POSITION"] - 1
    df["end_0based"] = df["POSITION"] + df["LENGTH"] - 1

    dfs.append(df)

zhunt_all = pd.concat(dfs, ignore_index=True)

print("Total Z-DNA predictions:", len(zhunt_all))
zhunt_all.head()

Total Z-DNA predictions: 1605859


,ID,POSITION,LENGTH,SEQUENCE,ZDNAGCRICHNESS,ZDNAGTRICHNESS,SCORE,SCOREPERC,LG,chrom,start_1based,end_1based,start_0based,end_0based
0,1,9629,51,GCACACACACACACACACACACACACACACACACACACACACACAC...,2.0,0.0,86.0,13.76,LG10,NC_043006.1,9629,9679,9628,9679
1,2,12275,23,ACACACACACACACACACACACA,0.0,0.0,33.0,12.00,LG10,NC_043006.1,12275,12297,12274,12297
2,3,12544,40,CACACACACACACACACACACACACACACACACACACACA,0.0,0.0,58.5,12.00,LG10,NC_043006.1,12544,12583,12543,12583
3,4,12617,17,ACACACACACACGCACA,12.5,0.0,46.0,23.00,LG10,NC_043006.1,12617,12633,12616,12633
4,5,12654,13,ACACACACACACA,0.0,0.0,18.0,12.00,LG10,NC_043006.1,12654,12666,12653,12666


In [10]:
zhunt_all.to_csv("zhunt_all_genome.csv", index=False)

In [11]:
zhunt_bed = zhunt_all[
    ["chrom", "start_0based", "end_0based", "LG", "SCORE", "SCOREPERC"]
].copy()

zhunt_bed.to_csv(
    "zhunt_all_genome.bed",
    sep="\t",
    header=False,
    index=False
)

zhunt_bed.head()

,chrom,start_0based,end_0based,LG,SCORE,SCOREPERC
0,NC_043006.1,9628,9679,LG10,86.0,13.76
1,NC_043006.1,12274,12297,LG10,33.0,12.00
2,NC_043006.1,12543,12583,LG10,58.5,12.00
3,NC_043006.1,12616,12633,LG10,46.0,23.00
4,NC_043006.1,12653,12666,LG10,18.0,12.00


In [12]:
print(zhunt_all.shape)
zhunt_all.head()

(1605859, 14)


,ID,POSITION,LENGTH,SEQUENCE,ZDNAGCRICHNESS,ZDNAGTRICHNESS,SCORE,SCOREPERC,LG,chrom,start_1based,end_1based,start_0based,end_0based
0,1,9629,51,GCACACACACACACACACACACACACACACACACACACACACACAC...,2.0,0.0,86.0,13.76,LG10,NC_043006.1,9629,9679,9628,9679
1,2,12275,23,ACACACACACACACACACACACA,0.0,0.0,33.0,12.00,LG10,NC_043006.1,12275,12297,12274,12297
2,3,12544,40,CACACACACACACACACACACACACACACACACACACACA,0.0,0.0,58.5,12.00,LG10,NC_043006.1,12544,12583,12543,12583
3,4,12617,17,ACACACACACACGCACA,12.5,0.0,46.0,23.00,LG10,NC_043006.1,12617,12633,12616,12633
4,5,12654,13,ACACACACACACA,0.0,0.0,18.0,12.00,LG10,NC_043006.1,12654,12666,12653,12666


In [13]:
zhunt_filtered = zhunt_all[zhunt_all["SCORE"] > 400].copy()

print("All Z-DNA predictions:", len(zhunt_all))
print("Filtered Z-DNA predictions SCORE > 400:", len(zhunt_filtered))

zhunt_filtered.head()

All Z-DNA predictions: 1605859
Filtered Z-DNA predictions SCORE > 400: 539


,ID,POSITION,LENGTH,SEQUENCE,ZDNAGCRICHNESS,ZDNAGTRICHNESS,SCORE,SCOREPERC,LG,chrom,start_1based,end_1based,start_0based,end_0based
2569,2570,3784591,137,TGTGTGTGTGCGTGCGTGTGTGTGTGCGTGCGTGTGTGTGTGCGTG...,23.529412,76.470588,556.0,32.705883,LG10,NC_043006.1,3784591,3784727,3784590,3784727
2590,2591,3803750,125,ACACACACACACACACACACACACGCACACACACACACACGCGCAC...,16.129032,0.000000,406.0,26.193546,LG10,NC_043006.1,3803750,3803874,3803749,3803874
3043,3044,4731946,113,ACACACGCACGCACACACACACGCACGTGCACGCACACACACGCAC...,33.035714,1.785714,575.0,41.071430,LG10,NC_043006.1,4731946,4732058,4731945,4732058
7330,7331,11484608,110,CGCACGCACGCACGCACACACGCACGCACGCACGCACGCACGCACG...,29.357798,0.000000,515.5,37.834862,LG10,NC_043006.1,11484608,11484717,11484607,11484717
12288,12289,19508921,107,TGTGTGTGTGTGTGTGCGTGCGTGCGTGCGTGCGTGCGTGCGTGCG...,43.396226,56.603774,665.0,50.188679,LG10,NC_043006.1,19508921,19509027,19508920,19509027


In [14]:
zhunt_filtered.to_csv("zhunt_all_genome_score_gt_400.csv", index=False)

zhunt_filtered_bed = zhunt_filtered[
    ["chrom", "start_0based", "end_0based", "LG", "SCORE", "SCOREPERC"]
].copy()

zhunt_filtered_bed.to_csv(
    "zhunt_all_genome_score_gt_400.bed",
    sep="\t",
    header=False,
    index=False
)

zhunt_filtered_bed.head()

,chrom,start_0based,end_0based,LG,SCORE,SCOREPERC
2569,NC_043006.1,3784590,3784727,LG10,556.0,32.705883
2590,NC_043006.1,3803749,3803874,LG10,406.0,26.193546
3043,NC_043006.1,4731945,4732058,LG10,575.0,41.071430
7330,NC_043006.1,11484607,11484717,LG10,515.5,37.834862
12288,NC_043006.1,19508920,19509027,LG10,665.0,50.188679


In [15]:
print(zhunt_filtered.shape)
zhunt_filtered.head()

(539, 14)


,ID,POSITION,LENGTH,SEQUENCE,ZDNAGCRICHNESS,ZDNAGTRICHNESS,SCORE,SCOREPERC,LG,chrom,start_1based,end_1based,start_0based,end_0based
2569,2570,3784591,137,TGTGTGTGTGCGTGCGTGTGTGTGTGCGTGCGTGTGTGTGTGCGTG...,23.529412,76.470588,556.0,32.705883,LG10,NC_043006.1,3784591,3784727,3784590,3784727
2590,2591,3803750,125,ACACACACACACACACACACACACGCACACACACACACACGCGCAC...,16.129032,0.000000,406.0,26.193546,LG10,NC_043006.1,3803750,3803874,3803749,3803874
3043,3044,4731946,113,ACACACGCACGCACACACACACGCACGTGCACGCACACACACGCAC...,33.035714,1.785714,575.0,41.071430,LG10,NC_043006.1,4731946,4732058,4731945,4732058
7330,7331,11484608,110,CGCACGCACGCACGCACACACGCACGCACGCACGCACGCACGCACG...,29.357798,0.000000,515.5,37.834862,LG10,NC_043006.1,11484608,11484717,11484607,11484717
12288,12289,19508921,107,TGTGTGTGTGTGTGTGCGTGCGTGCGTGCGTGCGTGCGTGCGTGCG...,43.396226,56.603774,665.0,50.188679,LG10,NC_043006.1,19508921,19509027,19508920,19509027


Для определения расположения Z-ДНК относительно генов используется таблица аннотации NCBI feature_table

In [16]:
feature_table = pd.read_csv(
    "GCF_006345805.1_ASM634580v1_feature_table.txt",
    sep="\t",
    low_memory=False
)

print(feature_table.columns.tolist())

['# feature', 'class', 'assembly', 'assembly_unit', 'seq_type', 'chromosome', 'genomic_accession', 'start', 'end', 'strand', 'product_accession', 'non-redundant_refseq', 'related_accession', 'name', 'symbol', 'GeneID', 'locus_tag', 'feature_interval_length', 'product_length', 'attributes']


Из аннотации извлекаются все гены с координатами, направлением транскрипции и идентификаторами GeneID

In [17]:
genes = feature_table[feature_table["# feature"] == "gene"].copy()

genes = genes[
    [
        "genomic_accession",
        "chromosome",
        "start",
        "end",
        "strand",
        "name",
        "symbol",
        "GeneID",
    ]
].copy()

genes = genes.dropna(subset=["genomic_accession", "start", "end", "strand"])

genes["start"] = genes["start"].astype(int)
genes["end"] = genes["end"].astype(int)

print("Genes:", len(genes))
genes.head()

Genes: 29784


,genomic_accession,chromosome,start,end,strand,name,symbol,GeneID
0,NC_042997.1,LG1,259763,260263,-,NaN,LOC115214395,115214395.0
3,NC_042997.1,LG1,1451649,1451767,+,NaN,LOC115220859,115220859.0
4,NC_042997.1,LG1,2502467,2502558,+,NaN,Trnak-uuu,115221612.0
6,NC_042997.1,LG1,2520268,3052735,-,NaN,LOC115219228,115219228.0
17,NC_042997.1,LG1,2779873,2808759,+,NaN,LOC118765644,118765644.0


Для каждого гена определяется промотор длиной 1000 п.н. вверх по течению от точки начала транскрипции (TSS)

In [18]:
promoters = genes.copy()

promoters["promoter_start"] = promoters.apply(
    lambda row: max(1, row["start"] - 1000) if row["strand"] == "+" else row["end"] + 1,
    axis=1
)

promoters["promoter_end"] = promoters.apply(
    lambda row: row["start"] - 1 if row["strand"] == "+" else row["end"] + 1000,
    axis=1
)

promoters = promoters[promoters["promoter_start"] <= promoters["promoter_end"]].copy()

print("Promoters:", len(promoters))
promoters.head()

Promoters: 29781


,genomic_accession,chromosome,start,end,strand,name,symbol,GeneID,promoter_start,promoter_end
0,NC_042997.1,LG1,259763,260263,-,NaN,LOC115214395,115214395.0,260264,261263
3,NC_042997.1,LG1,1451649,1451767,+,NaN,LOC115220859,115220859.0,1450649,1451648
4,NC_042997.1,LG1,2502467,2502558,+,NaN,Trnak-uuu,115221612.0,2501467,2502466
6,NC_042997.1,LG1,2520268,3052735,-,NaN,LOC115219228,115219228.0,3052736,3053735
17,NC_042997.1,LG1,2779873,2808759,+,NaN,LOC118765644,118765644.0,2778873,2779872


In [19]:
hits = []

for chrom, z_chr in zhunt_filtered.groupby("chrom"):
    p_chr = promoters[promoters["genomic_accession"] == chrom]

    for _, z in z_chr.iterrows():
        overlaps = p_chr[
            (p_chr["promoter_start"] <= z["end_1based"]) &
            (p_chr["promoter_end"] >= z["start_1based"])
        ]

        for _, p in overlaps.iterrows():
            hits.append({
                "chrom": chrom,
                "zdna_start": z["start_1based"],
                "zdna_end": z["end_1based"],
                "zdna_score": z["SCORE"],
                "promoter_start": p["promoter_start"],
                "promoter_end": p["promoter_end"],
                "gene_symbol": p["symbol"],
                "gene_name": p["name"],
                "GeneID": p["GeneID"],
                "strand": p["strand"],
            })

zhunt_promoters = pd.DataFrame(hits)

print("ZHunt hits in promoters:", len(zhunt_promoters))
zhunt_promoters.head()

ZHunt hits in promoters: 12


,chrom,zdna_start,zdna_end,zdna_score,promoter_start,promoter_end,gene_symbol,gene_name,GeneID,strand
0,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,NaN,115218751.0,-
1,NC_042998.1,156589991,156590116,616.5,156589077,156590076,LOC118762079,NaN,118762079.0,+
2,NC_043002.1,47167384,47167504,664.0,47166731,47167730,LOC115213476,NaN,115213476.0,+
3,NC_043003.1,27803275,27803406,680.5,27802819,27803818,LOC115214019,NaN,115214019.0,-
4,NC_043005.1,793941,794063,513.0,793801,794800,LOC118764748,NaN,118764748.0,-


In [20]:
zhunt_promoters.to_csv(
    "zhunt_predictions_in_promoters.csv",
    index=False
)

zhunt_promoters.head(20)

,chrom,zdna_start,zdna_end,zdna_score,promoter_start,promoter_end,gene_symbol,gene_name,GeneID,strand
0,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,NaN,115218751.0,-
1,NC_042998.1,156589991,156590116,616.5,156589077,156590076,LOC118762079,NaN,118762079.0,+
2,NC_043002.1,47167384,47167504,664.0,47166731,47167730,LOC115213476,NaN,115213476.0,+
3,NC_043003.1,27803275,27803406,680.5,27802819,27803818,LOC115214019,NaN,115214019.0,-
4,NC_043005.1,793941,794063,513.0,793801,794800,LOC118764748,NaN,118764748.0,-
5,NC_043006.1,41029285,41029355,457.0,41028656,41029655,LOC118765107,NaN,118765107.0,-
6,NC_043007.1,41062258,41062365,732.5,41061420,41062419,LOC115217605,NaN,115217605.0,+
7,NC_043010.1,67506953,67507040,526.5,67506486,67507485,LOC118766039,NaN,118766039.0,-
8,NC_043013.1,23687772,23687857,567.5,23687056,23688055,LOC115220919,NaN,115220919.0,+
9,NC_043014.1,14301181,14301357,1155.0,14301128,14302127,LOC115221623,NaN,115221623.0,+


In [21]:
genes[
    genes["GeneID"].isin(
        zhunt_promoters["GeneID"]
    )
].head(20)

,genomic_accession,chromosome,start,end,strand,name,symbol,GeneID
6228,NC_042997.1,LG1,183625806,183686935,-,NaN,LOC115218751,115218751.0
12551,NC_042998.1,LG2,156590077,156605899,+,NaN,LOC118762079,118762079.0
33521,NC_043002.1,LG6,47167731,47246521,+,NaN,LOC115213476,115213476.0
37608,NC_043003.1,LG7,27802333,27802818,-,NaN,LOC115214019,115214019.0
43984,NC_043005.1,LG9,771378,793800,-,NaN,LOC118764748,118764748.0
49255,NC_043006.1,LG10,41011002,41028655,-,NaN,LOC118765107,118765107.0
53670,NC_043007.1,LG11,41062420,41143796,+,NaN,LOC115217605,115217605.0
64390,NC_043010.1,LG14,67490178,67506485,-,NaN,LOC118766039,118766039.0
70446,NC_043013.1,LG17,23688056,23730170,+,NaN,LOC115220919,115220919.0
72579,NC_043014.1,LG18,14302128,14347381,+,NaN,LOC115221623,115221623.0


In [22]:
genes[
    genes["GeneID"].isin(
        zhunt_promoters["GeneID"]
    )
][
    [
        "GeneID",
        "symbol",
        "name"
    ]
]

,GeneID,symbol,name
6228,115218751.0,LOC115218751,NaN
12551,118762079.0,LOC118762079,NaN
33521,115213476.0,LOC115213476,NaN
37608,115214019.0,LOC115214019,NaN
43984,118764748.0,LOC118764748,NaN
49255,118765107.0,LOC118765107,NaN
53670,115217605.0,LOC115217605,NaN
64390,118766039.0,LOC118766039,NaN
70446,115220919.0,LOC115220919,NaN
72579,115221623.0,LOC115221623,NaN


In [23]:
feature_table[
    feature_table["GeneID"].isin(
        zhunt_promoters["GeneID"]
    )
].head(20)

,# feature,class,assembly,assembly_unit,seq_type,chromosome,genomic_accession,start,end,strand,product_accession,non-redundant_refseq,related_accession,name,symbol,GeneID,locus_tag,feature_interval_length,product_length,attributes
6228,gene,protein_coding,GCF_006345805.1,Primary Assembly,linkage group,LG1,NC_042997.1,183625806,183686935,-,NaN,NaN,NaN,NaN,LOC115218751,115218751.0,NaN,61130,NaN,NaN
6229,mRNA,NaN,GCF_006345805.1,Primary Assembly,linkage group,LG1,NC_042997.1,183625806,183686935,-,XM_029788664.2,NaN,XP_029644524.1,testis-specific serine/threonine-protein kinas...,LOC115218751,115218751.0,NaN,2701,2701.0,NaN
6230,mRNA,NaN,GCF_006345805.1,Primary Assembly,linkage group,LG1,NC_042997.1,183625806,183686935,-,XM_036507550.1,NaN,XP_036363443.1,testis-specific serine/threonine-protein kinas...,LOC115218751,115218751.0,NaN,2751,2751.0,NaN
6231,mRNA,NaN,GCF_006345805.1,Primary Assembly,linkage group,LG1,NC_042997.1,183625806,183686935,-,XM_036507561.1,NaN,XP_036363454.1,testis-specific serine/threonine-protein kinas...,LOC115218751,115218751.0,NaN,2705,2705.0,NaN
6232,mRNA,NaN,GCF_006345805.1,Primary Assembly,linkage group,LG1,NC_042997.1,183625806,183686935,-,XM_036507558.1,NaN,XP_036363451.1,testis-specific serine/threonine-protein kinas...,LOC115218751,115218751.0,NaN,2755,2755.0,NaN
6233,CDS,with_protein,GCF_006345805.1,Primary Assembly,linkage group,LG1,NC_042997.1,183626610,183627521,-,XP_029644524.1,NaN,XM_029788664.2,testis-specific serine/threonine-protein kinas...,LOC115218751,115218751.0,NaN,912,303.0,NaN
6234,CDS,with_protein,GCF_006345805.1,Primary Assembly,linkage group,LG1,NC_042997.1,183626610,183627521,-,XP_036363443.1,NaN,XM_036507550.1,testis-specific serine/threonine-protein kinas...,LOC115218751,115218751.0,NaN,912,303.0,NaN
6235,CDS,with_protein,GCF_006345805.1,Primary Assembly,linkage group,LG1,NC_042997.1,183626610,183627521,-,XP_036363451.1,NaN,XM_036507558.1,testis-specific serine/threonine-protein kinas...,LOC115218751,115218751.0,NaN,912,303.0,NaN
6236,CDS,with_protein,GCF_006345805.1,Primary Assembly,linkage group,LG1,NC_042997.1,183626610,183627521,-,XP_036363454.1,NaN,XM_036507561.1,testis-specific serine/threonine-protein kinas...,LOC115218751,115218751.0,NaN,912,303.0,NaN
12551,gene,lncRNA,GCF_006345805.1,Primary Assembly,linkage group,LG2,NC_042998.1,156590077,156605899,+,NaN,NaN,NaN,NaN,LOC118762079,118762079.0,NaN,15823,NaN,NaN


In [24]:
promoter_gene_names = []

for gene_id in zhunt_promoters["GeneID"].unique():

    tmp = feature_table[
        (feature_table["GeneID"] == gene_id) &
        (feature_table["name"].notna())
    ]

    if len(tmp):
        promoter_gene_names.append(
            tmp.iloc[0][
                [
                    "GeneID",
                    "symbol",
                    "name"
                ]
            ]
        )

promoter_gene_names = pd.DataFrame(promoter_gene_names)

promoter_gene_names

,GeneID,symbol,name
6229,115218751.0,LOC115218751,testis-specific serine/threonine-protein kinas...
12552,118762079.0,LOC118762079,uncharacterized LOC118762079
33522,115213476.0,LOC115213476,uncharacterized LOC115213476
37609,115214019.0,LOC115214019,uncharacterized LOC115214019
43985,118764748.0,LOC118764748,uncharacterized LOC118764748
49256,118765107.0,LOC118765107,uncharacterized LOC118765107
53671,115217605.0,LOC115217605,organic cation transporter protein-like
64391,118766039.0,LOC118766039,uncharacterized LOC118766039
70447,115220919.0,LOC115220919,A disintegrin and metalloproteinase with throm...
72580,115221623.0,LOC115221623,"galactosylceramide sulfotransferase-like, tran..."


In [25]:
gene_names = (
    feature_table[
        feature_table["name"].notna()
    ][
        ["GeneID", "name"]
    ]
    .drop_duplicates()
)

final_hits = zhunt_promoters.merge(
    gene_names,
    on="GeneID",
    how="left"
)

final_hits.head()

,chrom,zdna_start,zdna_end,zdna_score,promoter_start,promoter_end,gene_symbol,gene_name,GeneID,strand,name
0,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,NaN,115218751.0,-,testis-specific serine/threonine-protein kinas...
1,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,NaN,115218751.0,-,testis-specific serine/threonine-protein kinas...
2,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,NaN,115218751.0,-,testis-specific serine/threonine-protein kinas...
3,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,NaN,115218751.0,-,testis-specific serine/threonine-protein kinas...
4,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,NaN,115218751.0,-,testis-specific serine/threonine-protein kinas...


In [26]:
group_file = final_hits[
    [
        "chrom",
        "zdna_start",
        "zdna_end",
        "zdna_score",
        "promoter_start",
        "promoter_end",
        "gene_symbol",
        "GeneID",
        "strand",
        "name",
    ]
].copy()

group_file.to_csv(
    "octopus_zdna_promoters_for_group.tsv",
    sep="\t",
    index=False
)

group_file

,chrom,zdna_start,zdna_end,zdna_score,promoter_start,promoter_end,gene_symbol,GeneID,strand,name
0,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,115218751.0,-,testis-specific serine/threonine-protein kinas...
1,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,115218751.0,-,testis-specific serine/threonine-protein kinas...
2,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,115218751.0,-,testis-specific serine/threonine-protein kinas...
3,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,115218751.0,-,testis-specific serine/threonine-protein kinas...
4,NC_042997.1,183687631,183687698,474.5,183686936,183687935,LOC115218751,115218751.0,-,testis-specific serine/threonine-protein kinas...
5,NC_042998.1,156589991,156590116,616.5,156589077,156590076,LOC118762079,118762079.0,+,uncharacterized LOC118762079
6,NC_043002.1,47167384,47167504,664.0,47166731,47167730,LOC115213476,115213476.0,+,uncharacterized LOC115213476
7,NC_043002.1,47167384,47167504,664.0,47166731,47167730,LOC115213476,115213476.0,+,uncharacterized protein LOC115213476
8,NC_043003.1,27803275,27803406,680.5,27802819,27803818,LOC115214019,115214019.0,-,uncharacterized LOC115214019
9,NC_043003.1,27803275,27803406,680.5,27802819,27803818,LOC115214019,115214019.0,-,uncharacterized protein LOC115214019


Пересечение координат предсказанных Z-DNA участков и промоторов с использованием библиотеки bioframe

In [28]:
!pip install bioframe -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.2 MB/s eta 0:00:00


In [30]:
import bioframe
import pandas as pd

zhunt_all_bed = pd.DataFrame({
    "chrom": zhunt_all["chrom"].astype(str),
    "start": zhunt_all["start_0based"].astype(int),
    "end": zhunt_all["end_0based"].astype(int),
    "score": zhunt_all["SCORE"].astype(float),
    "scoreperc": zhunt_all["SCOREPERC"].astype(float),
    "sequence": zhunt_all["SEQUENCE"].astype(str),
})

promoters_bed = pd.DataFrame({
    "chrom": promoters["genomic_accession"].astype(str),
    "start": (promoters["promoter_start"].astype(int) - 1),
    "end": promoters["promoter_end"].astype(int),
    "gene_symbol": promoters["symbol"].astype(str),
    "gene_name": promoters["name"].astype(str),
    "GeneID": promoters["GeneID"],
    "strand": promoters["strand"].astype(str),
})

overlaps_all = bioframe.overlap(
    zhunt_all_bed,
    promoters_bed,
    how="inner",
    suffixes=("_zdna", "_promoter")
)

print("ZHunt hits in promoters without SCORE filter:", len(overlaps_all))
overlaps_all.head()

ZHunt hits in promoters without SCORE filter: 17919


,chrom_zdna,start_zdna,end_zdna,score_zdna,scoreperc_zdna,sequence_zdna,chrom_promoter,start_promoter,end_promoter,gene_symbol_promoter,gene_name_promoter,GeneID_promoter,strand_promoter
0,NC_043022.1,19612410,19612465,81.0,12.0,ACACACACACACACACACACACACACACACACACACACACACACAC...,NC_043022.1,19612423,19613423,LOC115224863,nan,115224863.0,+
1,NC_043022.1,20181042,20181059,24.0,12.0,CACACACACACACACAC,NC_043022.1,20181051,20182051,LOC115224864,nan,115224864.0,+
2,NC_043022.1,20321355,20321369,19.5,12.0,TGTGTGTGTGTGTG,NC_043022.1,20321365,20322365,LOC118768023,nan,118768023.0,-
3,NC_043022.1,133381,133403,31.5,12.0,ACACACACACACACACACACAC,NC_043022.1,132635,133635,LOC115224653,nan,115224653.0,+
4,NC_043022.1,133403,133418,21.0,12.0,CACACACACACACAC,NC_043022.1,132635,133635,LOC115224653,nan,115224653.0,+


In [31]:
zhunt_filtered_bed = zhunt_all_bed[zhunt_all_bed["score"] > 400].copy()

overlaps_filtered = bioframe.overlap(
    zhunt_filtered_bed,
    promoters_bed,
    how="inner",
    suffixes=("_zdna", "_promoter")
)

print("ZHunt hits in promoters SCORE > 400:", len(overlaps_filtered))
overlaps_filtered.head()

ZHunt hits in promoters SCORE > 400: 12


,chrom_zdna,start_zdna,end_zdna,score_zdna,scoreperc_zdna,sequence_zdna,chrom_promoter,start_promoter,end_promoter,gene_symbol_promoter,gene_name_promoter,GeneID_promoter,strand_promoter
0,NC_043010.1,67506952,67507040,526.5,48.413792,ACACACACGCACGCACGCACGCACGCACGCACGCACGCACGCACGC...,NC_043010.1,67506485,67507485,LOC118766039,nan,118766039.0,-
1,NC_043003.1,27803274,27803406,680.5,41.557251,CACACACACACGCACGCGCACGCACGCACGCACGCGCACGCACGCA...,NC_043003.1,27802818,27803818,LOC115214019,nan,115214019.0,-
2,NC_043014.1,14301180,14301357,1155.0,52.499996,TGCACACGCGCGCGCACACACGCACGCACGCGCACACACGCACGCA...,NC_043014.1,14301127,14302127,LOC115221623,nan,115221623.0,+
3,NC_042997.1,183687630,183687698,474.5,56.656719,ACGTGCGTGCGTGCGTGCGTGCGTGCGTGCGTGCGTGCGTGCGTGC...,NC_042997.1,183686935,183687935,LOC115218751,nan,115218751.0,-
4,NC_042998.1,156589990,156590116,616.5,39.456001,TGCACACGCACGCACGCACGCACGCACGCACGCACGCACGCACGCA...,NC_042998.1,156589076,156590076,LOC118762079,nan,118762079.0,+


Для найденных пересечений добавляются названия генов и формируется итоговый файл для группового анализа

In [34]:
# Добавляем нормальные названия генов из строк mRNA/ncRNA/CDS
gene_names = (
    feature_table[feature_table["name"].notna()][["GeneID", "name"]]
    .drop_duplicates(subset=["GeneID"])
)

def make_group_format(overlaps: pd.DataFrame) -> pd.DataFrame:
    result = overlaps.merge(
        gene_names,
        left_on="GeneID_promoter",
        right_on="GeneID",
        how="left",
    )

    result["gene_name"] = result["name"].fillna(result["gene_name_promoter"])
    result["gene_name"] = result["gene_name"].replace("nan", "")

    group_format = pd.DataFrame({
        "chrom_zdna": result["chrom_zdna"],
        "zdna_start": result["start_zdna"] + 1,
        "zdna_end": result["end_zdna"],
        "score": result["score_zdna"],
        "chrom_promoter": result["chrom_promoter"],
        "promoter_start": result["start_promoter"] + 1,
        "promoter_end": result["end_promoter"],
        "gene_symbol": result["gene_symbol_promoter"],
        "strand": result["strand_promoter"],
        "gene_name": result["gene_name"],
    })

    return group_format

group_all = make_group_format(overlaps_all)
group_filtered = make_group_format(overlaps_filtered)

group_all.to_csv("octopus_zdna_promoters_all.csv", index=False)
group_filtered.to_csv("octopus_zdna_promoters_score_gt_400.csv", index=False)

print("All promoter hits:", len(group_all))
print("Filtered promoter hits:", len(group_filtered))

group_filtered.head()

All promoter hits: 17919
Filtered promoter hits: 12


,chrom_zdna,zdna_start,zdna_end,score,chrom_promoter,promoter_start,promoter_end,gene_symbol,strand,gene_name
0,NC_043010.1,67506953,67507040,526.5,NC_043010.1,67506486,67507485,LOC118766039,-,uncharacterized LOC118766039
1,NC_043003.1,27803275,27803406,680.5,NC_043003.1,27802819,27803818,LOC115214019,-,uncharacterized LOC115214019
2,NC_043014.1,14301181,14301357,1155.0,NC_043014.1,14301128,14302127,LOC115221623,+,"galactosylceramide sulfotransferase-like, tran..."
3,NC_042997.1,183687631,183687698,474.5,NC_042997.1,183686936,183687935,LOC115218751,-,testis-specific serine/threonine-protein kinas...
4,NC_042998.1,156589991,156590116,616.5,NC_042998.1,156589077,156590076,LOC118762079,+,uncharacterized LOC118762079


In [37]:
!zip -r results.zip \
    *.csv \
    *.bed \
    *.tsv

  adding: octopus_zdna_promoters_all.csv (deflated 82%)
  adding: octopus_zdna_promoters_score_gt_400.csv (deflated 57%)
  adding: zhunt_all_genome.csv (deflated 81%)
  adding: zhunt_all_genome_score_gt_400.csv (deflated 77%)
  adding: zhunt_predictions_in_promoters.csv (deflated 58%)
  adding: zhunt_all_genome.bed (deflated 80%)
  adding: zhunt_all_genome_score_gt_400.bed (deflated 65%)
  adding: octopus_zdna_promoters_all.tsv (deflated 82%)
  adding: octopus_zdna_promoters_for_group.tsv (deflated 80%)
  adding: octopus_zdna_promoters_score_gt_400.tsv (deflated 58%)


In [38]:
from google.colab import files

files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Поиск SMC/RAD21 в аннотации

In [39]:
feature_table = pd.read_csv(
    "GCF_006345805.1_ASM634580v1_feature_table.txt",
    sep="\t",
    low_memory=False
)

keywords = "SMC1|SMC2|SMC3|SMC4|RAD21|structural maintenance of chromosomes|cohesin"

candidates = feature_table[
    feature_table["name"].str.contains(
        keywords,
        case=False,
        na=False,
        regex=True
    )
][
    ["# feature", "name", "symbol", "product_accession", "GeneID"]
]

candidates.drop_duplicates()

,# feature,name,symbol,product_accession,GeneID
13470,mRNA,non-structural maintenance of chromosomes elem...,LOC115228770,XM_029799269.2,115228770.0
13471,CDS,non-structural maintenance of chromosomes elem...,LOC115228770,XP_029655129.1,115228770.0
14836,mRNA,structural maintenance of chromosomes flexible...,LOC115209208,XM_029777461.2,115209208.0
14837,mRNA,structural maintenance of chromosomes flexible...,LOC115209208,XM_029777459.2,115209208.0
14838,mRNA,structural maintenance of chromosomes flexible...,LOC115209208,XM_036500832.1,115209208.0
14839,CDS,structural maintenance of chromosomes flexible...,LOC115209208,XP_029633321.1,115209208.0
14840,CDS,structural maintenance of chromosomes flexible...,LOC115209208,XP_029633319.1,115209208.0
14841,CDS,structural maintenance of chromosomes flexible...,LOC115209208,XP_036356725.1,115209208.0
21368,mRNA,double-strand-break repair protein rad21 homol...,LOC115211098,XM_029779989.2,115211098.0
21369,mRNA,double-strand-break repair protein rad21 homol...,LOC115211098,XM_029779991.2,115211098.0


In [41]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.2 MB/s eta 0:00:00


In [42]:
from Bio import SeqIO

target_keywords = {
    "SMC1": ["structural maintenance of chromosomes protein 1", "SMC1"],
    "SMC2": ["structural maintenance of chromosomes protein 2", "SMC2"],
    "SMC3": ["structural maintenance of chromosomes protein 3", "SMC3"],
    "SMC4": ["structural maintenance of chromosomes protein 4", "SMC4"],
    "RAD21": ["double-strand-break repair protein rad21", "RAD21", "cohesin complex component RAD21"],
}

records = list(SeqIO.parse("GCF_006345805.1_ASM634580v1_protein.faa", "fasta"))

selected = []

for gene_name, keys in target_keywords.items():
    matches = []
    for record in records:
        desc = record.description.lower()
        if any(key.lower() in desc for key in keys):
            matches.append(record)

    print(gene_name, len(matches))
    for m in matches[:5]:
        print(" ", m.id, m.description[:120])

    if matches:
        best = max(matches, key=lambda r: len(r.seq))
        best.id = f"Octopus_sinensis_{gene_name}"
        best.name = ""
        best.description = ""
        selected.append(best)

SeqIO.write(selected, "octopus_sinensis_SMC_RAD21.fasta", "fasta")

print("Saved:", len(selected), "sequences")

SMC1 1
  XP_029635846.1 XP_029635846.1 structural maintenance of chromosomes protein 1A [Octopus sinensis]
SMC2 1
  XP_029641022.1 XP_029641022.1 structural maintenance of chromosomes protein 2 [Octopus sinensis]
SMC3 3
  XP_029645024.1 XP_029645024.1 structural maintenance of chromosomes protein 3 isoform X1 [Octopus sinensis]
  XP_029645025.2 XP_029645025.2 structural maintenance of chromosomes protein 3 isoform X2 [Octopus sinensis]
  XP_029656809.1 XP_029656809.1 LOW QUALITY PROTEIN: structural maintenance of chromosomes protein 3-like [Octopus sinensis]
SMC4 1
  XP_029636692.1 XP_029636692.1 structural maintenance of chromosomes protein 4 [Octopus sinensis]
RAD21 3
  XP_029635849.1 XP_029635849.1 double-strand-break repair protein rad21 homolog isoform X1 [Octopus sinensis]
  XP_029635851.1 XP_029635851.1 double-strand-break repair protein rad21 homolog isoform X2 [Octopus sinensis]
  XP_036357782.1 XP_036357782.1 double-strand-break repair protein rad21 homolog isoform X1 [Octopu

In [43]:
genes_with_zdna = (
    group_all[["gene_symbol", "gene_name"]]
    .drop_duplicates()
    .sort_values(["gene_name", "gene_symbol"])
)

genes_with_zdna.to_csv("octopus_genes_with_zdna_in_promoters.csv", index=False)

print(len(genes_with_zdna))
genes_with_zdna.head(30)

8131


,gene_symbol,gene_name
7535,LOC115208790,
8407,LOC115208829,
7435,LOC115208842,
8468,LOC115208849,
7282,LOC115208850,
7483,LOC115208855,
7608,LOC115208859,
7673,LOC115208870,
7733,LOC115208871,
7406,LOC115208874,


In [44]:
target_keywords = [
    "F-box DNA helicase",
    "GATA zinc finger protein 16",
    "ATF-4",
    "REC8",
    "histone H2A",
    "double-stranded RNA-specific editase",
    "ADAR",
]

mask = group_all["gene_name"].str.contains(
    "|".join(target_keywords),
    case=False,
    na=False,
    regex=True
)

target_zdna_genes = group_all[mask].copy()

print(target_zdna_genes.shape)
target_zdna_genes[
    ["gene_symbol", "gene_name", "chrom_promoter", "promoter_start", "promoter_end", "strand"]
].drop_duplicates()

(8, 10)


,gene_symbol,gene_name,chrom_promoter,promoter_start,promoter_end,strand
1747,LOC115213971,cyclic AMP-dependent transcription factor ATF-4,NC_043003.1,47175227,47176226,+
3526,LOC115223636,"histone H2A, sperm-like",NC_043019.1,639218,640217,-
3801,LOC115223515,histone H2A deubiquitinase MYSM1-like,NC_043019.1,22121827,22122826,+
15271,LOC115216616,histone H2A,NC_043006.1,63883672,63884671,+
16083,LOC115215479,histone H2A-like,NC_043005.1,92661201,92662200,+


In [46]:
interesting = [
    "ATF-4",
    "histone H2A",
    "histone H2A-like",
    "histone H2A deubiquitinase MYSM1-like"
]

feature_table[
    feature_table["name"].str.contains(
        "|".join(interesting),
        case=False,
        na=False
    )
][[
    "# feature",
    "name",
    "symbol",
    "product_accession",
    "GeneID"
]]

,# feature,name,symbol,product_accession,GeneID
12214,mRNA,histone H2A.V,LOC115232386,XM_029802242.2,115232386.0
12215,CDS,histone H2A.V,LOC115232386,XP_029658102.1,115232386.0
16157,mRNA,histone H2A-like,LOC115209111,XM_029777238.2,115209111.0
16158,CDS,histone H2A-like,LOC115209111,XP_029633098.1,115209111.0
38355,mRNA,cyclic AMP-dependent transcription factor ATF-4,LOC115213971,XM_029782976.2,115213971.0
...,...,...,...,...,...
109573,CDS,histone H2A-like,LOC115231062,XP_029657010.1,115231062.0
111060,mRNA,"histone H2A-beta, sperm-like",LOC115231511,XM_029801524.1,115231511.0
111061,CDS,"histone H2A-beta, sperm-like",LOC115231511,XP_029657384.1,115231511.0
111075,mRNA,"histone H2A-beta, sperm-like",LOC115231514,XM_029801527.1,115231514.0


In [51]:
target_zdna_genes.columns

Index(['chrom_zdna', 'zdna_start', 'zdna_end', 'score', 'chrom_promoter',
       'promoter_start', 'promoter_end', 'gene_symbol', 'strand', 'gene_name'],
      dtype='object')

In [52]:
# Берём только найденные нами гены-кандидаты с Z-DNA в промоторе
target_symbols = target_zdna_genes["gene_symbol"].dropna().unique()

cds_targets = feature_table[
    (feature_table["symbol"].isin(target_symbols)) &
    (feature_table["# feature"] == "CDS") &
    (feature_table["product_accession"].notna())
][
    ["symbol", "name", "product_accession", "GeneID"]
].drop_duplicates()

cds_targets

,symbol,name,product_accession,GeneID
38356,LOC115213971,cyclic AMP-dependent transcription factor ATF-4,XP_029638836.1,115213971.0
47011,LOC115215479,histone H2A-like,XP_029640503.2,115215479.0
50185,LOC115216616,histone H2A,XP_029641956.1,115216616.0
81776,LOC115223636,"histone H2A, sperm-like",XP_029650167.1,115223636.0
82999,LOC115223515,histone H2A deubiquitinase MYSM1-like,XP_029650002.1,115223515.0


In [53]:
target_accessions = set(cds_targets["product_accession"])

selected = []

for record in SeqIO.parse(
    "GCF_006345805.1_ASM634580v1_protein.faa",
    "fasta"
):
    acc = record.id.split()[0]

    if acc in target_accessions:
        selected.append(record)

SeqIO.write(
    selected,
    "octopus_zdna_promoter_genes_proteins.fasta",
    "fasta"
)

print("Saved:", len(selected))

Saved: 5


In [54]:
from google.colab import files

files.download("octopus_zdna_promoter_genes_proteins.fasta")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Распределение Z-Hunt по Exons, Introns, Promoters, Downstream, Intergenic**

In [64]:
exons = feature_table[
    feature_table["# feature"].isin(["exon", "CDS"])
].copy()

exons = exons[
    ["genomic_accession", "start", "end"]
].dropna()

exons["start"] = exons["start"].astype(int)
exons["end"] = exons["end"].astype(int)

print("Exon/CDS regions:", len(exons))

Exon/CDS regions: 36112


In [65]:
import bioframe
import pandas as pd

zdna = pd.DataFrame({
    "chrom": zhunt_filtered["chrom"].astype(str),
    "start": zhunt_filtered["start_0based"].astype(int),
    "end": zhunt_filtered["end_0based"].astype(int),
})
zdna["id"] = range(len(zdna))

genes_bed = pd.DataFrame({
    "chrom": genes["genomic_accession"].astype(str),
    "start": genes["start"].astype(int) - 1,
    "end": genes["end"].astype(int),
})

exons_bed = pd.DataFrame({
    "chrom": exons["genomic_accession"].astype(str),
    "start": exons["start"].astype(int) - 1,
    "end": exons["end"].astype(int),
})

promoters_bed_simple = pd.DataFrame({
    "chrom": promoters["genomic_accession"].astype(str),
    "start": promoters["promoter_start"].astype(int) - 1,
    "end": promoters["promoter_end"].astype(int),
})

downstream_bed = pd.DataFrame({
    "chrom": downstream["genomic_accession"].astype(str),
    "start": downstream["downstream_start"].astype(int) - 1,
    "end": downstream["downstream_end"].astype(int),
})

# Интроны: gene regions минус exon/CDS regions
genes_merged = bioframe.merge(genes_bed)
exons_merged = bioframe.merge(exons_bed)
introns_bed = bioframe.subtract(genes_merged, exons_merged)

def overlap_ids(regions):
    ov = bioframe.overlap(zdna, regions, how="inner")
    return set(ov["id"].dropna().astype(int))

exon_ids = overlap_ids(exons_bed)
intron_ids = overlap_ids(introns_bed)
promoter_ids = overlap_ids(promoters_bed_simple)
downstream_ids = overlap_ids(downstream_bed)
gene_ids = overlap_ids(genes_bed)

intergenic_ids = set(zdna["id"]) - gene_ids - promoter_ids - downstream_ids

distribution = pd.DataFrame({
    "Участок": [
        "Exons/CDS",
        "Introns",
        "Promoters (1000 up from TSS)",
        "Downstream (200 bp)",
        "Intergenic",
    ],
    "Число предсказаний Z-Hunt": [
        len(exon_ids),
        len(intron_ids),
        len(promoter_ids),
        len(downstream_ids),
        len(intergenic_ids),
    ],
})

distribution["Доля предсказаний Z-Hunt"] = (
    distribution["Число предсказаний Z-Hunt"] / len(zdna)
)

distribution

,Участок,Число предсказаний Z-Hunt,Доля предсказаний Z-Hunt
0,Exons/CDS,123,0.228200
1,Introns,75,0.139147
2,Promoters (1000 up from TSS),12,0.022263
3,Downstream (200 bp),0,0.000000
4,Intergenic,334,0.619666


In [56]:
downstream = genes.copy()

downstream["downstream_start"] = downstream.apply(
    lambda row: row["end"] + 1 if row["strand"] == "+" else max(1, row["start"] - 200),
    axis=1
)

downstream["downstream_end"] = downstream.apply(
    lambda row: row["end"] + 200 if row["strand"] == "+" else row["start"] - 1,
    axis=1
)

downstream = downstream[
    downstream["downstream_start"] <= downstream["downstream_end"]
].copy()

print("Downstream:", len(downstream))

Downstream: 29781


In [57]:
gene_regions = genes[
    ["genomic_accession", "start", "end"]
].copy()

In [58]:
gene_regions

,genomic_accession,start,end
0,NC_042997.1,259763,260263
3,NC_042997.1,1451649,1451767
4,NC_042997.1,2502467,2502558
6,NC_042997.1,2520268,3052735
17,NC_042997.1,2779873,2808759
...,...,...,...
113338,NC_006353.1,7565,8908
113340,NC_006353.1,8905,9201
113344,NC_006353.1,9346,10483
113346,NC_006353.1,10476,10988


In [60]:
def count_hits(zhunt_df, regions, chrom_col="genomic_accession"):

    zdna = pd.DataFrame({
        "chrom": zhunt_df["chrom"],
        "start": zhunt_df["start_0based"],
        "end": zhunt_df["end_0based"]
    })

    reg = pd.DataFrame({
        "chrom": regions[chrom_col],
        "start": regions["start"] - 1,
        "end": regions["end"]
    })

    overlaps = bioframe.overlap(
        zdna,
        reg,
        how="inner"
    )

    return len(overlaps)

In [61]:
results = {
    "Genes": count_hits(zhunt_filtered, gene_regions),
    "Exons": count_hits(zhunt_filtered, exons),
    "Promoters": len(overlaps_filtered),
}

In [62]:
results

{'Genes': 202, 'Exons': 0, 'Promoters': 12}

In [63]:
len(zhunt_filtered)

539

In [66]:
!wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/006/345/805/GCF_006345805.1_ASM634580v1/GCF_006345805.1_ASM634580v1_genomic.fna.gz

!gunzip GCF_006345805.1_ASM634580v1_genomic.fna.gz

!ls -lh GCF_006345805.1_ASM634580v1_genomic.fna

--2026-06-10 21:14:09--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/006/345/805/GCF_006345805.1_ASM634580v1/GCF_006345805.1_ASM634580v1_genomic.fna.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.12, 130.14.250.13, 2607:f220:41e:250::7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 830255842 (792M) [application/x-gzip]
Saving to: ‘GCF_006345805.1_ASM634580v1_genomic.fna.gz’

GCF_006345805.1_ASM 100%[===================>] 791.79M  59.5MB/s    in 14s     

2026-06-10 21:14:23 (57.7 MB/s) - ‘GCF_006345805.1_ASM634580v1_genomic.fna.gz’ saved [830255842/830255842]

-rw-r--r-- 1 root root 2.6G Feb  2  2022 GCF_006345805.1_ASM634580v1_genomic.fna


In [67]:
!grep "^>" GCF_006345805.1_ASM634580v1_genomic.fna | head

>NC_042997.1 Octopus sinensis linkage group LG1, ASM634580v1, whole genome shotgun sequence
>NC_042998.1 Octopus sinensis linkage group LG2, ASM634580v1, whole genome shotgun sequence
>NC_042999.1 Octopus sinensis linkage group LG3, ASM634580v1, whole genome shotgun sequence
>NC_043000.1 Octopus sinensis linkage group LG4, ASM634580v1, whole genome shotgun sequence
>NC_043001.1 Octopus sinensis linkage group LG5, ASM634580v1, whole genome shotgun sequence
>NC_043002.1 Octopus sinensis linkage group LG6, ASM634580v1, whole genome shotgun sequence
>NC_043003.1 Octopus sinensis linkage group LG7, ASM634580v1, whole genome shotgun sequence
>NC_043004.1 Octopus sinensis linkage group LG8, ASM634580v1, whole genome shotgun sequence
>NC_043005.1 Octopus sinensis linkage group LG9, ASM634580v1, whole genome shotgun sequence
>NC_043006.1 Octopus sinensis linkage group LG10, ASM634580v1, whole genome shotgun sequence


In [68]:
!grep -c "^>" GCF_006345805.1_ASM634580v1_genomic.fna

13516


In [69]:
!pip install biopython -q

In [73]:
import re
import pandas as pd
from Bio import SeqIO
from time import time

pattern_plus = re.compile(r"(?:G{3,5}[ATGC]{1,7}){3,}G{3,5}")
pattern_minus = re.compile(r"(?:C{3,5}[ATGC]{1,7}){3,}C{3,5}")

target_accessions = {
    f"NC_{42997+i:06d}.1"
    for i in range(30)
}

results = []
t0 = time()

for record in SeqIO.parse(
    "GCF_006345805.1_ASM634580v1_genomic.fna",
    "fasta"
):
    chrom = record.id

    if chrom not in target_accessions:
        continue

    seq = str(record.seq).upper()
    seq_len = len(seq)

    chrom_count_before = len(results)
    print(f"Processing {chrom}, length={seq_len:,}")

    for m in pattern_plus.finditer(seq):
        results.append({
            "chrom": chrom,
            "start_1based": m.start() + 1,
            "end_1based": m.end(),
            "strand": "+",
            "sequence": m.group(),
        })

    for m in pattern_minus.finditer(seq):
        results.append({
            "chrom": chrom,
            "start_1based": seq_len - m.end() + 1,
            "end_1based": seq_len - m.start(),
            "strand": "-",
            "sequence": m.group(),
        })

    chrom_count = len(results) - chrom_count_before
    print(f"  G4 found: {chrom_count:,}")

g4 = pd.DataFrame(results)

g4["start_0based"] = g4["start_1based"] - 1
g4["end_0based"] = g4["end_1based"]

print("Total G4:", len(g4))
print("Time:", round(time() - t0, 2), "sec")

g4.to_csv("octopus_g4_predictions.csv", index=False)

g4.head()

Processing NC_042997.1, length=213,406,131
  G4 found: 4,786
Processing NC_042998.1, length=207,928,902
  G4 found: 4,665
Processing NC_042999.1, length=174,423,281
  G4 found: 3,661
Processing NC_043000.1, length=164,836,235
  G4 found: 3,674
Processing NC_043001.1, length=135,913,566
  G4 found: 2,929
Processing NC_043002.1, length=120,472,873
  G4 found: 3,104
Processing NC_043003.1, length=117,540,060
  G4 found: 3,029
Processing NC_043004.1, length=107,269,306
  G4 found: 2,662
Processing NC_043005.1, length=109,703,381
  G4 found: 2,435
Processing NC_043006.1, length=105,892,736
  G4 found: 2,642
Processing NC_043007.1, length=77,502,973
  G4 found: 2,136
Processing NC_043008.1, length=79,792,878
  G4 found: 1,503
Processing NC_043009.1, length=72,296,471
  G4 found: 1,837
Processing NC_043010.1, length=68,403,164
  G4 found: 2,296
Processing NC_043011.1, length=67,901,367
  G4 found: 2,030
Processing NC_043012.1, length=59,466,961
  G4 found: 1,744
Processing NC_043013.1, length

,chrom,start_1based,end_1based,strand,sequence,start_0based,end_0based
0,NC_042997.1,58196,58244,+,GGGGTTAGGGTTAGTTAGGGTTAGCTAGGGTTAGGGGTGGGGGGAAGGG,58195,58244
1,NC_042997.1,265843,265864,+,GGGAGAGGGGGGAGGGGTAGGG,265842,265864
2,NC_042997.1,278395,278419,+,GGGGTGGTGGGGGGGACTTTTTGGG,278394,278419
3,NC_042997.1,346285,346302,+,GGGTGGGATGGGTGGGGG,346284,346302
4,NC_042997.1,349458,349480,+,GGGGTGGGTTAACGAGGGAGGGG,349457,349480


In [74]:
feature_table["# feature"].value_counts().head(20)

,count
# feature,
CDS,36112
mRNA,36099
gene,29784
ncRNA,7750
tRNA,1949
misc_RNA,1298
rRNA,371


In [75]:
g4_bed = pd.DataFrame({
    "chrom": g4["chrom"].astype(str),
    "start": g4["start_0based"].astype(int),
    "end": g4["end_0based"].astype(int),
})
g4_bed["id"] = range(len(g4_bed))

cds = feature_table[
    feature_table["# feature"] == "CDS"
].copy()

cds = cds[
    ["genomic_accession", "start", "end"]
].dropna()

cds["start"] = cds["start"].astype(int)
cds["end"] = cds["end"].astype(int)

cds_bed = pd.DataFrame({
    "chrom": cds["genomic_accession"].astype(str),
    "start": cds["start"].astype(int) - 1,
    "end": cds["end"].astype(int),
})

genes_bed = pd.DataFrame({
    "chrom": genes["genomic_accession"].astype(str),
    "start": genes["start"].astype(int) - 1,
    "end": genes["end"].astype(int),
})

promoters_bed_simple = pd.DataFrame({
    "chrom": promoters["genomic_accession"].astype(str),
    "start": promoters["promoter_start"].astype(int) - 1,
    "end": promoters["promoter_end"].astype(int),
})

downstream_bed = pd.DataFrame({
    "chrom": downstream["genomic_accession"].astype(str),
    "start": downstream["downstream_start"].astype(int) - 1,
    "end": downstream["downstream_end"].astype(int),
})

genes_merged = bioframe.merge(genes_bed)
cds_merged = bioframe.merge(cds_bed)
introns_bed = bioframe.subtract(genes_merged, cds_merged)

def overlap_ids(query_bed, regions_bed):
    ov = bioframe.overlap(query_bed, regions_bed, how="inner")
    return set(ov["id"].dropna().astype(int))

g4_cds_ids = overlap_ids(g4_bed, cds_bed)
g4_intron_ids = overlap_ids(g4_bed, introns_bed)
g4_promoter_ids = overlap_ids(g4_bed, promoters_bed_simple)
g4_downstream_ids = overlap_ids(g4_bed, downstream_bed)
g4_gene_ids = overlap_ids(g4_bed, genes_bed)

g4_intergenic_ids = set(g4_bed["id"]) - g4_gene_ids - g4_promoter_ids - g4_downstream_ids

g4_distribution = pd.DataFrame({
    "Участок": [
        "Exons/CDS",
        "Introns",
        "Promoters (1000 up from TSS)",
        "Downstream (200 bp)",
        "Intergenic",
    ],
    "Число квадруплексов": [
        len(g4_cds_ids),
        len(g4_intron_ids),
        len(g4_promoter_ids),
        len(g4_downstream_ids),
        len(g4_intergenic_ids),
    ],
})

g4_distribution["Доля квадруплексов"] = (
    g4_distribution["Число квадруплексов"] / len(g4_bed)
)

g4_distribution

,Участок,Число квадруплексов,Доля квадруплексов
0,Exons/CDS,21275,0.337891
1,Introns,7145,0.113478
2,Promoters (1000 up from TSS),933,0.014818
3,Downstream (200 bp),135,0.002144
4,Intergenic,33748,0.535989


In [77]:
promoters_2000 = genes.copy()

promoters_2000["promoter_start"] = promoters_2000.apply(
    lambda row: max(1, row["start"] - 2000) if row["strand"] == "+" else row["end"] + 1,
    axis=1
)

promoters_2000["promoter_end"] = promoters_2000.apply(
    lambda row: row["start"] - 1 if row["strand"] == "+" else row["end"] + 2000,
    axis=1
)

promoters_2000 = promoters_2000[
    promoters_2000["promoter_start"] <= promoters_2000["promoter_end"]
].copy()

print("Promoters 2000:", len(promoters_2000))

Promoters 2000: 29781


In [78]:
downstream_2000 = genes.copy()

downstream_2000["downstream_start"] = downstream_2000.apply(
    lambda row: row["end"] + 1 if row["strand"] == "+" else max(1, row["start"] - 2000),
    axis=1
)

downstream_2000["downstream_end"] = downstream_2000.apply(
    lambda row: row["end"] + 2000 if row["strand"] == "+" else row["start"] - 1,
    axis=1
)

downstream_2000 = downstream_2000[
    downstream_2000["downstream_start"] <= downstream_2000["downstream_end"]
].copy()

print("Downstream 2000:", len(downstream_2000))

Downstream 2000: 29781


In [79]:
promoters_2000_bed = pd.DataFrame({
    "chrom": promoters_2000["genomic_accession"].astype(str),
    "start": promoters_2000["promoter_start"].astype(int) - 1,
    "end": promoters_2000["promoter_end"].astype(int),
})

downstream_2000_bed = pd.DataFrame({
    "chrom": downstream_2000["genomic_accession"].astype(str),
    "start": downstream_2000["downstream_start"].astype(int) - 1,
    "end": downstream_2000["downstream_end"].astype(int),
})

In [80]:
g4_promoter_2000_ids = overlap_ids(g4_bed, promoters_2000_bed)
g4_downstream_2000_ids = overlap_ids(g4_bed, downstream_2000_bed)

g4_intergenic_2000_ids = (
    set(g4_bed["id"])
    - g4_gene_ids
    - g4_promoter_2000_ids
    - g4_downstream_2000_ids
)

g4_distribution_2000 = pd.DataFrame({
    "Участок": [
        "Exons/CDS",
        "Introns",
        "Promoters (2000 up from TSS)",
        "Downstream (2000 bp)",
        "Intergenic",
    ],
    "Число квадруплексов": [
        len(g4_cds_ids),
        len(g4_intron_ids),
        len(g4_promoter_2000_ids),
        len(g4_downstream_2000_ids),
        len(g4_intergenic_2000_ids),
    ],
})

g4_distribution_2000["Доля квадруплексов"] = (
    g4_distribution_2000["Число квадруплексов"] / len(g4_bed)
)

g4_distribution_2000

,Участок,Число квадруплексов,Доля квадруплексов
0,Exons/CDS,21275,0.337891
1,Introns,7145,0.113478
2,Promoters (2000 up from TSS),1674,0.026587
3,Downstream (2000 bp),1359,0.021584
4,Intergenic,32343,0.513674


In [ ]:
g4["start_0based"] = g4["start_1based"] - 1
g4["end_0based"] = g4["end_1based"]

g4.to_csv(
    "octopus_g4_predictions.csv",
    index=False
)

In [76]:
from google.colab import files

files.download("octopus_g4_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>